# 🏆 Aircheck Workshop: Hackathon

**AIRCHECK Workshop 2026** · Train a model on the WDR91 DEL screen, rank the hackathon
library, and submit your top 200.

This is the hands-on notebook cut down to what the hackathon needs. Same pipeline, same
code, with the teaching detours removed: no fingerprint fusion, no discussion sections, no
extra steps, and no evaluation against labels, because the library you screen here does not
carry any.

---

## 📄 Where this comes from

This notebook is adapted from this article:**[Enabling Open Machine Learning of
Deoxyribonucleic Acid-Encoded Library Selections to Accelerate the Discovery of Small Molecule
Protein Binders**](https://pmc.ncbi.nlm.nih.gov/articles/PMC12557371/)**, and the WDR91 data it
uses is the dataset released in AIRCHECK website: https://aircheck.ai/


The article's codes are on GitHub at
[jimmyjbling/SGC-DEL-ML-WDR91](https://github.com/jimmyjbling/SGC-DEL-ML-WDR91/tree/main).

---

# 🏆 Hackathon Details

You already have everything you need. The notebook above is a complete, working pipeline,
and the hackathon is about making it *yours*.

## The format of the hackathon section

| | |
|---|---|
| Time | **one hour** |
| Team size | **2** |
| Leaderboard | picks up your latest file every **5 minutes**, from each user's dedicated output folder |
| Mentors | on the floor throughout, please use them |

## The data

The main files are **`train_wdr91_full.parquet`** for training and
**`test_hackathon_full_WithoutLabel.parquet`** for test: every fingerprint, every row,
nothing removed. The other variants are that same data with columns or rows dropped, made
for your convenience so you are not waiting on a 650 MB read when your model only needs
ECFP4. Pick whichever suits your plan. They are all cuts of the same dataset.

### Training files, pick one

| File | Rows | Actives | Fingerprints | Size | Why you would pick it |
|---|---|---|---|---|---|
| `train_wdr91_full.parquet` | 375,595 | 28,778 (7.7%) | all nine | 656 MB | **the main training file**, everything, unmodified |
| `train_wdr91_balanced.parquet` | 57,556 | 28,778 (50%) | all nine | 101 MB | every active kept, negatives sampled 1:1, trains far faster |
| `train_wdr91_ECFP4_FCFP4_TOPTOR_ATOMPAIR.parquet` | 375,595 | 28,778 (7.7%) | four | 257 MB | all rows, four fingerprints to compare |
| `train_wdr91_ECFP4.parquet` | 375,595 | 28,778 (7.7%) | ECFP4 only | 49 MB | quickest to load, and ECFP4 is the fingerprint the notebook above used |

### Test files, score one

**The main test file is `test_hackathon_full_WithoutLabel.parquet`.** The two smaller ones
hold exactly the same compounds with fewer fingerprint columns, again for convenience.

| File | Fingerprints | Size |
|---|---|---|
| `test_hackathon_full_WithoutLabel.parquet` | all nine | 626 MB |
| `test_hackathon_ECFP4_FCFP4_TOPTOR_ATOMPAIR_WithoutLabel.parquet` | four | 229 MB |
| `test_hackathon_ECFP4_WithoutLabel.parquet` | ECFP4 only | 52 MB |

**Where to find them:** both files read straight off the workshop volume.

```python
df_train = pd.read_parquet("/Volumes/uhn_workshop/lab/input/Train/train_wdr91_full.parquet")
df_test  = pd.read_parquet("/Volumes/uhn_workshop/lab/input/Test/test_hackathon_full_WithoutLabel.parquet")
```

**Known actives in the test set:** **252** compounds carry `LABEL == 1`.

## What you submit

A CSV named after your team, in exactly the format the submission cell writes:

| Column | Contents |
|---|---|
| `SMILES` | the compound |
| `Prediction_Score` | your model's score for it |

- **200 rows**, ordered best first
- Saved to your own output folder
- Set `TEAM_NAME` and `SUBMISSION_DIR` at the top of the submission cell

Rank, keep the top 200 and save into your own folder in one line,
with your own user number in place of `02`, leading zero and all:

```python
prediction_df_sorted.sort_values("Prediction_Score", ascending=False).head(200)[["SMILES", "Prediction_Score"]].to_csv("/Volumes/uhn_workshop/lab/output_user_02/Team1.csv", index=False)
```

Or set `SUBMISSION_DIR = Path("/Volumes/uhn_workshop/lab/output_user_02")` in the submission
cell and let it do the ranking, the trim to 200 and the format checks for you.

## How you are scored

**Metric: hit@K, at K = 50, 100 and 200.**

Your submission is a ranked list. `hit@K` counts how many of the 252 known actives appear in
your top K rows. Reporting three cutoffs means the order inside your 200 matters, not only
which compounds you picked: two teams nominating the same set score differently if one ranks
the actives higher.

## Ideas worth trying in an hour

Roughly in order of effort:

1. **Change the fingerprint.** `selected_fps` in Section 3, one word, and ECFP4 is not always the winner. Load a different one by editing `LOAD_FINGERPRINTS` in Section 2.
2. **Tune harder.** Section 6 tries five configurations by hand. Widen the grid, or hand it to `RandomizedSearchCV`.
3. **Change the class balance.** The optional cell in Section 3 varies negatives per positive. The full training file is 7.66% active; the library you are ranking is far rarer than that.
4. **Try a different model.** Section 4 defines four alternatives, each a one-line swap.
5. **Train an ensemble.** Fit one model per fingerprint and average their scores, penalising the compounds they disagree about. A dozen lines, and it usually beats a single model at the sharp end of the list.
6. **Split by group.** A random split flatters a DEL model, because most compounds share a building block with something in the training folds. A group-aware split will *lower* your validation score and may well *raise* your leaderboard score.

> **Worth heeding.** Every one of those can be judged *before* you submit, with
> cross-validation on the training data. The library has no labels, so submitting blind and
> reading the leaderboard back is the slowest way to learn anything.

Good luck. The pipeline already works. Go and make it better.

---

# 📦 Section 1 · Install and Import Dependencies

In this section we install the packages the workshop needs for chemical data processing
and machine learning.

In [ ]:
# --- run-time tracking -----------------------------------------------------
# Times every cell, so the last cell can report how long the whole notebook took.
# Harmless outside Jupyter/Colab, and costs nothing to run.
import time as _time

ECHO_CELL_TIME = False    # True prints each cell's own time under its output

CELL_TIMES = []
_timer_state = {}

try:
    from IPython import get_ipython

    def _timer_pre(info):
        _timer_state["t0"] = _time.perf_counter()
        _timer_state["src"] = getattr(info, "raw_cell", "")

    def _timer_post(result):
        t0 = _timer_state.pop("t0", None)
        if t0 is None:
            return
        src = _timer_state.pop("src", "")
        first = next((l.strip() for l in src.split("\n") if l.strip()), "")
        taken = _time.perf_counter() - t0
        CELL_TIMES.append((taken, first[:70]))
        if ECHO_CELL_TIME:
            print(f"[cell {len(CELL_TIMES):>2}  {taken:6.2f}s]")

    _ip = get_ipython()
    if _ip is not None and not _timer_state.get("registered"):
        _ip.events.register("pre_run_cell", _timer_pre)
        _ip.events.register("post_run_cell", _timer_post)
        _timer_state["registered"] = True
except Exception:
    pass          # timing is a convenience; never let it break the notebook

NOTEBOOK_STARTED = _time.time()
# ---------------------------------------------------------------------------

# This notebook runs in Google Colab, on Databricks, and from a local clone of the
# repository. It uses the repository if it is already on disk and clones it if not,
# so the paths below come out the same in all three places.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ShagReza/Aircheck-Workshop-2026.git"
REPO_NAME = "Aircheck-Workshop-2026"

IN_COLAB = "google.colab" in sys.modules
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ


def find_repo_root(start):
    """Walk up from `start` looking for the repository root, or None."""
    candidate = Path(start).resolve()
    while candidate != candidate.parent:
        if (candidate / "requirements.txt").exists():
            return candidate
        candidate = candidate.parent
    return None


# Already on disk? A local clone, or a Databricks Git folder, will be found here.
REPO_ROOT = find_repo_root(Path.cwd())

if REPO_ROOT is None:
    # Colab, or a runtime where the repository is not checked out: clone it.
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO_ROOT = Path(REPO_NAME).resolve()

DATA_DIR = REPO_ROOT / "sample_data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

environment = "Colab" if IN_COLAB else "Databricks" if IN_DATABRICKS else "locally"
print(f"Running in {environment}")
print(f"Repository root: {REPO_ROOT}")
print(f"Data files:      {sorted(p.name for p in DATA_DIR.glob('*.parquet'))}")

In [ ]:
# Install every package the workshop needs, as listed in requirements.txt.
# In Colab and Databricks, install into the current runtime.
# Locally, assume the virtual environment has already been prepared.

import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_COLAB or IN_DATABRICKS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(REPO_ROOT / "requirements.txt"),
        ],
        check=True,
    )
    print("Requirements installed.")
else:
    print("Local run - install requirements with:")
    print(f"    python -m pip install -r {REPO_ROOT / 'requirements.txt'}")

In [ ]:
# Import libraries:
import os
import sys

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem

# Make the repository's src/ package importable, then load the workshop helpers.
# REPO_ROOT was worked out in the bootstrap cell, so this behaves identically in
# Colab (where the repo was cloned) and locally.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.metrics import hits_at_k, enrichment_at_k, screening_table
from src.plots import (plot_cv_metrics, plot_score_distribution, plot_pr_and_roc,
                       plot_enrichment_curve, plot_threshold_tradeoff,
                       plot_molecule_grid)

# The K values we report everywhere: how many actives sit in the top K compounds.
KS = (20, 50, 100, 200, 500)

print("Workshop helpers loaded from", REPO_ROOT / "src")


---

# 📂 Section 2 · Load the Data

Two files, and they play different roles.

| Role | File | What it is |
|---|---|---|
| **Training data** | `train_wdr91_full.parquet` | the WDR91 DEL screen, labelled. The model learns from this, and cross-validation on it is how you judge your choices. |
| **Screening library** | `test_hackathon_full_WithoutLabel.parquet` | 341,622 compounds with no labels. You rank these and submit the top 200. |

The rule that matters: **every decision you make has to be made on the training data.** The
library has no labels, so it cannot tell you whether a choice helped. Cross-validate,
decide, then rank once.

Off Databricks the notebook falls back to the small committed samples, using
`sample-screen.parquet` as the library so it is unlabelled there too.

In [ ]:
# Which training file. The full one is the real screen, 375,595 compounds at 7.66% active.
# The balanced one keeps every active and samples the inactives 1:1, so it is 57,556 rows and
# trains roughly six times faster, which is useful while you are still trying things out.
TRAIN_FILE = "Train/train_wdr91_full.parquet"
# TRAIN_FILE = "Train/train_wdr91_balanced.parquet"

# The library you rank. It has no LABEL column: that is what the leaderboard holds.
TEST_FILE = "Test/test_hackathon_full_WithoutLabel.parquet"

# Where the files come from:
#   "azure"  the workshop volume. This is what counts, and what the leaderboard scores.
#            If the volume is not reachable this stops rather than quietly using something else.
#   "local"  the small samples in this repository. For checking the notebook runs. Results
#            from them are NOT submittable.
#   "auto"   volume if it is there, samples if it is not.
SOURCE = "azure"

import pyarrow.parquet as pq

VOLUME = Path("/Volumes/uhn_workshop/lab/input")

if SOURCE not in ("azure", "local", "auto"):
    raise ValueError(f"SOURCE must be 'azure', 'local' or 'auto', not {SOURCE!r}")


def find_data(volume_name, sample_name):
    """Return (path, where) for one dataset, honouring SOURCE."""
    on_volume = VOLUME / volume_name
    sample = DATA_DIR / sample_name

    if SOURCE in ("azure", "auto") and on_volume.exists():
        return on_volume, "Azure volume"

    if SOURCE == "azure":
        raise FileNotFoundError(
            f"SOURCE is 'azure' but {on_volume} is not there. Check the volume path, or "
            "set SOURCE = 'local' to check the notebook runs on the small samples.")

    if sample.exists():
        return sample, "local sample"

    raise FileNotFoundError(
        f"Could not find this dataset. Looked on the volume at {on_volume} "
        f"and in the repository at {sample}.")


TRAIN_PATH, TRAIN_SOURCE = find_data(TRAIN_FILE, "sample-train.parquet")
TEST_PATH, TEST_SOURCE = find_data(TEST_FILE, "sample-screen.parquet")

# The files hold nine fingerprints. Reading all nine is what exhausts a cluster, so only the
# ones listed here are loaded. Add a name and re-run this cell to work with more.
ALL_FINGERPRINTS = ["ECFP4", "ECFP6", "FCFP4", "FCFP6", "MACCS",
                    "RDK", "AVALON", "ATOMPAIR", "TOPTOR"]
LOAD_FINGERPRINTS = ["ECFP4", "RDK"]

BATCH_ROWS = 50_000


def load_meta(path):
    """SMILES and LABEL, whichever the file has. Cheap: these are scalars, not arrays."""
    available = pq.ParquetFile(path).schema_arrow.names
    wanted = [c for c in ("SMILES", "LABEL") if c in available]
    return pq.read_table(path, columns=wanted).to_pandas()


def load_fingerprints(path, names):
    """One (n_molecules, n_bits) uint8 matrix per fingerprint, built a batch at a time.

    Parquet stores a fingerprint as list<int32>, and pandas cannot hold that as a matrix, so
    it builds one small array per molecule: 375,595 objects and over 3 GB for a single column
    of the full file. Reading Arrow batches straight into a matrix allocated once keeps the
    peak to the size of the result plus one batch.
    """
    parquet = pq.ParquetFile(path)
    rows = parquet.metadata.num_rows
    present = [n for n in names if n in parquet.schema_arrow.names]
    missing = [n for n in names if n not in present]
    if missing:
        print(f"not in {Path(path).name}, skipped: {', '.join(missing)}")

    peek = next(parquet.iter_batches(batch_size=1, columns=present))
    matrices = {n: np.empty((rows, len(peek.column(n)[0])), dtype=np.uint8) for n in present}

    at = 0
    for batch in parquet.iter_batches(batch_size=BATCH_ROWS, columns=present):
        size = batch.num_rows
        for name in present:
            block = batch.column(name).flatten().to_numpy(zero_copy_only=False).reshape(size, -1)
            if block.max() > 255:
                raise ValueError(
                    f"{name} holds a count above 255, which will not fit in the uint8 "
                    "matrix. Widen the dtype in load_fingerprints before using this file.")
            matrices[name][at:at + size] = block
        at += size
    return matrices


df_train = load_meta(TRAIN_PATH)
df_test = load_meta(TEST_PATH)

TRAIN_FP = load_fingerprints(TRAIN_PATH, LOAD_FINGERPRINTS)
TEST_FP = load_fingerprints(TEST_PATH, LOAD_FINGERPRINTS)

for name, frame, fps, source, path, role in [
        ("df_train", df_train, TRAIN_FP, TRAIN_SOURCE, TRAIN_PATH, "training data"),
        ("df_test", df_test, TEST_FP, TEST_SOURCE, TEST_PATH, "screening library")]:
    if "LABEL" in frame.columns:
        actives = int(frame["LABEL"].sum())
        balance = f"actives: {actives:>6,} ({actives / len(frame):>6.2%})"
    else:
        balance = "unlabelled                "
    held = sum(m.nbytes for m in fps.values()) / 1e9
    print(f"{name:<9} {len(frame):>7,} rows   {balance}   {role}")
    print(f"{'':<9} {source}: {path}")
    print(f"{'':<9} fingerprints {', '.join(fps)}  holding {held:.2f} GB")

if "local sample" in (TRAIN_SOURCE, TEST_SOURCE):
    print()
    print("NOTE: running on the sample files, not the workshop data. Fine for checking the")
    print("      notebook runs, but do not submit a ranking built from these.")

## Training set

In [ ]:
# Display first few rows of the train dataset
df_train.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_train.columns.tolist()
print(column_names_list)

## The screening library

The compounds you will rank. **No `LABEL` column**: these are the ones the leaderboard
scores you on, so the answers are not in the file. That is the difference between this and
the hands-on notebook, and the reason this notebook never prints a test score.

In [ ]:
# Display first few rows of the test dataset
df_test.head()

In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_test.columns.tolist()
print(column_names_list)

---

# 🧬 Section 3 · Building the Feature Matrix

A molecule is a graph, not a row of numbers. Some models work on that graph directly, but a
gradient-boosted tree like LightGBM needs a fixed-length numeric row, and that is what a
**molecular fingerprint** gives it: each position records whether, or how often, a
particular substructure appears in the molecule.

The files already carry nine fingerprints, computed for you:

| Fingerprint | Bits | What it encodes |
|---|---|---|
| `ECFP4`, `ECFP6` | 2048 | circular substructures around each atom, radius 2 and 3 |
| `FCFP4`, `FCFP6` | 2048 | the same idea, but atoms grouped by chemical *function* |
| `MACCS` | 167 | 167 hand-written yes/no structural questions |
| `RDK` | 2048 | paths through the molecular graph |
| `AVALON` | 2048 | a mixed set of substructure and path features |
| `ATOMPAIR` | 2048 | pairs of atoms and the distance between them |
| `TOPTOR` | 2048 | torsions: short four-atom fragments |

Each entry in those columns is already a NumPy array of counts, so building a feature matrix
is just stacking them into rows. We default to **ECFP4**, the most widely used of the set.

**Only `ECFP4` and `RDK` are loaded.** All nine sit in the files, but reading every one of
them is what exhausts a cluster's memory, and two is enough to show everything this
notebook does. `LOAD_FINGERPRINTS` in Section 2 is where you change that.

In [ ]:
# The fingerprints actually loaded in Section 2. ALL_FINGERPRINTS names the nine the files
# hold; anything not loaded is not in memory and cannot be chosen here.
fingerprint_columns = LOAD_FINGERPRINTS
selected_fps = 'ECFP4'  # any one of LOAD_FINGERPRINTS

# Feature matrices, one per role.
TrainData = TRAIN_FP[selected_fps]
TestData = TEST_FP[selected_fps]

# Only the training data has labels. The library does not, which is the whole point of it.
TrainLabel = df_train['LABEL']

print(f"Fingerprint in use: {selected_fps}")
print(f"  TrainData {str(TrainData.shape):>16}   training data, split into folds in Section 5")
print(f"  TestData  {str(TestData.shape):>16}   the library you rank, unlabelled")
print()
actives = int(TrainLabel.sum())
print(f"  training labels: {actives:,} active / {len(TrainLabel) - actives:,} inactive")

# What does a fingerprint matrix actually look like?
print()
print(f"First 5 molecules, first 12 of {TrainData.shape[1]} bits:")
display(pd.DataFrame(TrainData[:5, :12],
                     columns=[f"bit_{i}" for i in range(12)]).astype(int))

print(f"Only {(TrainData > 0).mean():.1%} of all entries are non-zero - a fingerprint is a "
      f"sparse description, mostly recording which substructures are ABSENT.")

## Optional · Changing how much negative data you train on

This sample is balanced 50/50, which is convenient but not what a real DEL screen looks
like. There, inactives outnumber actives by orders of magnitude. When you point this
notebook at the full dataset you will have far more negatives than positives, and how many
of them you keep becomes a real decision.

The amount is set as a **ratio: negatives per positive.** `1.0` is balanced, `5.0` keeps
five inactives for every active, `0.5` keeps half as many inactives as actives. Every
positive is always kept, because actives are the scarce and expensive part of the data.

In [ ]:
# OPTIONAL - works on copies; df_train, TrainData and TrainLabel are untouched.
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold

from src.features import balance_summary

# Negatives per positive. 1.0 is balanced; above 1.0 needs more inactives than this
# balanced sample has, so those rows come back capped - on the real data they will not.
ratios = [0.25, 0.5, 1.0, 3.0]

print("What each requested ratio does to the training set:")
print(balance_summary(df_train, ratios).to_string(index=False))

print()
print("And what it does to 3-fold cross-validated performance:")
def resample_rows(labels, ratio, random_state=42):
    """Row positions holding `ratio` negatives per positive, every positive kept.

    Positions rather than a sub-dataframe, because the fingerprints live in a matrix now and
    the two have to stay aligned. src.features.resample_negatives cannot be used here: it
    ends with reset_index(drop=True), so the rows it returns can no longer be matched to
    rows of the matrix.
    """
    labels = np.asarray(labels)
    positives = np.flatnonzero(labels == 1)
    negatives = np.flatnonzero(labels == 0)
    wanted = int(round(len(positives) * ratio))
    if wanted > len(negatives):
        print(f"  ! ratio {ratio:g} needs {wanted} negatives but only {len(negatives)} "
              f"exist - using all of them")
        wanted = len(negatives)
    rng = np.random.default_rng(random_state)
    keep = np.concatenate([positives, rng.choice(negatives, size=wanted, replace=False)])
    rng.shuffle(keep)
    return keep


rows = []
for ratio in ratios:
    keep = resample_rows(df_train["LABEL"], ratio)
    scores = cross_validate(
        LGBMClassifier(random_state=42, verbose=-1),
        TRAIN_FP[selected_fps][keep], df_train["LABEL"].to_numpy()[keep],
        cv=StratifiedKFold(3, shuffle=True, random_state=42),
        scoring=["roc_auc", "average_precision"])
    rows.append({"ratio": ratio,
                 "rows used": len(keep),
                 "AUC-ROC": round(float(scores["test_roc_auc"].mean()), 3),
                 "avg precision": round(float(scores["test_average_precision"].mean()), 3)})

print(pd.DataFrame(rows).to_string(index=False))

print()
print(f"df_train unchanged : {len(df_train)} rows, {int(df_train['LABEL'].sum())} actives")
print(f"TrainData unchanged: {TrainData.shape}")

# ---------------------------------------------------------------------------
# TO ACTUALLY TRAIN ON A REBALANCED SET, do it explicitly and re-run from Section 4:
#
#     keep = resample_rows(df_train["LABEL"], ratio=5.0)
#     TrainData = TRAIN_FP[selected_fps][keep]
#     TrainLabel = df_train["LABEL"].to_numpy()[keep]
# ---------------------------------------------------------------------------

---

# ⚙️ Section 4 · Define the ML Model

## First: are we predicting a number or a category?

- **Regression** predicts a *number*. How enriched is this compound? What sequencing count
  would we expect?
- **Classification** predicts a *category*, normally with a probability attached. Is this
  compound active, yes or no?

Our training file carries both. `RawCount` is the raw sequencing readout, and `LABEL` is
`1` exactly when that count is above zero. 

**We classify**, for two reasons. The label is what a chemist acts on: you either put a
compound on the plate or you do not. And counts are noisy: the gap between 6 and 9 reads
says much more about sequencing depth than about binding.

> **When regression would be the better choice:** if you trust your enrichment values, a
> regression model ranks the strong binders above the marginal ones instead of lumping them
> into one "active" bucket. Note that only the training file has `RawCount` here, so we
> could not score such a model on the test set anyway.
>
> Almost everything else in this notebook (the splitting, the cross-validation, the
> ranking, the screen) would stay exactly the same. Only the model class and the metrics
> change.

## Which kind of classifier?

Every classifier draws a boundary between actives and inactives. They differ in the *shape*
of boundary they are able to draw, and in what that costs.

| Family | Example | The idea in one line | Watch out for |
|---|---|---|---|
| **Linear** | `LogisticRegression` | weigh each feature, add them up, squash the total into a probability | can only draw a straight boundary; wants scaled inputs |
| **Margin-based** | `SVC` | find the boundary with the widest possible gap between the classes | scales badly, slow well before 4,000 × 2,048 |
| **Bagged trees** | `RandomForestClassifier` | grow hundreds of trees on random slices and average them | large models, and rarely the strongest on this kind of data |
| **Neural network** | `MLPClassifier` | stacked weighted sums with non-linearities between them | hungry for data and tuning; seldom beats boosting on tabular data |

The fifth family is **boosted trees**, trees grown one at a time, each correcting the
errors of those before it. That is LightGBM, and it is what we will use.

The next cell defines all four, one line each, showing the few settings you would actually
reach for. Every value shown is the library default, so they behave exactly as the bare
constructor would. The end of the cell says where to plug one in.

In [ ]:
# Four alternatives to LightGBM, one line each. The values shown are the library
# defaults, so these behave exactly as `LogisticRegression()` would - they are spelled
# out only so you can see which knobs exist. Nothing is fitted in this cell.
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

logistic_model = LogisticRegression(C=1.0, penalty="l2", max_iter=1000, random_state=42)
svm_model = SVC(C=1.0, kernel="rbf", gamma="scale", probability=True, random_state=42)
forest_model = RandomForestClassifier(n_estimators=100, max_depth=None, n_jobs=-1, random_state=42)
mlp_model = MLPClassifier(hidden_layer_sizes=(100,), alpha=1e-4, max_iter=200, random_state=42)

alternative_models = {
    "Logistic regression": logistic_model,    # C: smaller means a simpler model
    "Support vector machine": svm_model,      # C and kernel; "linear" is far faster
    "Random forest": forest_model,            # n_estimators, max_depth
    "Neural network (MLP)": mlp_model,        # hidden_layer_sizes, e.g. (256, 64)
}

for name, estimator in alternative_models.items():
    print(f"{name:<24} {type(estimator).__name__}")

print()
print("All four answer to .fit(X, y) and .predict_proba(X), which is what makes")
print("swapping one in a small change rather than a rewrite.")

# To use one instead of LightGBM, replace the constructor in the two places a model is
# built - train_and_validate() in Section 5, train_final_model() in Section 7 - then
# re-run from there. Change both, so what you validate is what you screen with.

## LightGBM

Light Gradient Boosting Machine is a fast, scalable gradient boosting framework. It builds
decision trees iteratively, each one correcting the errors of the ones before it. Unlike
older gradient boosting implementations it uses a histogram-based approach, which speeds up
training considerably on large datasets, and it handles categorical features directly.

> **Why it suits this problem.** Fingerprints are wide, sparse, tabular count vectors, and
> gradient boosting is usually the strongest family on tabular data, and the hardest to beat
> without a lot of tuning. This is the same reasoning laid out in Section 2 of the ML
> introduction notebook.

In [ ]:
from lightgbm import LGBMClassifier

# Initialize model with detailed hyperparameters using default values
model = LGBMClassifier(
    n_estimators=100,  # Number of boosting iterations (trees)
    n_jobs=1,  # Number of parallel jobs (1 for no parallelism)
    learning_rate=0.1,  # Learning rate
    max_depth=-1,  # No limit on maximum depth of trees
    min_child_samples=20,  # Minimum samples at leaf node
    reg_lambda=0.0,  # L2 regularization (no regularization)
    reg_alpha=0.0,  # L1 regularization (no regularization)
    num_leaves=31,  # Number of leaves in each tree
    max_bin=255,  # Maximum number of bins
    subsample=1.0,  # Subsample ratio for training data (use all data)
    colsample_bytree=1.0,  # Subsample ratio for features (use all features)
    random_state=42,  # Random seed for reproducibility
    boosting_type='gbdt',  # Boosting type (Gradient Boosting Decision Tree)
    min_split_gain=0.0,  # Minimum loss reduction required to make a further partition
    verbose=-1,  # Quieten LightGBM's per-tree logging
)

# Model is now initialized with default hyperparameters

---

# 📏 Metrics · What Are We Actually Measuring?

Before we train anything, we should agree on how we will judge it. Pick the wrong metric and
you will confidently choose the wrong model.

Every classification metric is built from four counts. For one compound the model either
flags it or it does not, and it either is active or it is not:

|  | Predicted inactive | Predicted active |
|---|---|---|
| **Actually inactive** | true negative | **false positive**, an assay run for nothing |
| **Actually active** | **false negative**, a hit we never tested | true positive |

The two errors do not cost the same. A false positive wastes one well on a plate. A false
negative may be the compound the whole campaign was looking for.

## Metrics that need a yes/no decision

These take the model's hard prediction, which means they depend on where you put the
threshold. All of them appear in the cross-validation output in the next section.

| Metric | What it asks | Blind spot |
|---|---|---|
| **Accuracy** | what fraction did I get right? | useless when one class is rare, see below |
| **Precision** | of the ones I flagged, how many were real? | says nothing about the hits you missed |
| **Recall** | of the real actives, how many did I find? | flagging everything gives perfect recall |
| **F1** | the balance between precision and recall | one number hides which of the two is failing |
| **MCC** | agreement across all four counts at once | harder to explain to a non-specialist |
| **Cohen's kappa** | how much better than guessing at the same rate? | same idea as MCC, slightly different scale |

MCC and Cohen's kappa are the trustworthy single numbers on imbalanced data: both stay near zero
for a model that is really just exploiting the class balance.

## Metrics that use the ranking

These ignore the threshold and look at the ordering, which is closer to what a screen does.

| Metric | What it asks |
|---|---|
| **ROC-AUC** | pick one active and one inactive at random: how often is the active scored higher? |
| **Average precision (PR-AUC)** | precision averaged across every recall level |
| **hit@K** | of the top K compounds, how many are actually active? |
| **Enrichment** | how many times better than picking K at random? |

## Which ones matter for this workshop

Actives are rare in a screening library, usually well under 1%, and we will only ever assay
the top of the list. That combination decides everything:

- **Ignore accuracy.** A model that calls everything inactive scores over 99%.
- **ROC-AUC flatters.** With inactives dominating the comparison, it stays high even when
  the top of the list is poor.
- **Trust average precision, hit@K and enrichment.** They ask about the rare positives and
  about the part of the list you can afford to test, which is the actual decision.

The next cell shows all of this on made-up numbers, before any real model exists.

In [ ]:
# A demonstration on INVENTED data - no model is trained here. We fake three
# rankings over 5,000 compounds with 9 actives, a rarity typical of a screen, and
# see how each metric reacts.
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             matthews_corrcoef, cohen_kappa_score,
                             roc_auc_score, average_precision_score)

rng = np.random.default_rng(7)
n_compounds, n_actives = 5000, 9

# Scatter the actives at random positions. If they sat at the top of the array,
# a model that scores everything the same would appear to "find" them.
y_mock = np.zeros(n_compounds, dtype=int)
y_mock[rng.choice(n_compounds, n_actives, replace=False)] = 1

# three different "models", none of them real
scores = {
    "Calls everything inactive": np.full(n_compounds, 0.01),
    "Random ranking": rng.random(n_compounds),
    "A decent model": np.where(y_mock == 1,
                               rng.normal(0.70, 0.18, n_compounds),
                               rng.normal(0.20, 0.12, n_compounds)).clip(0, 1),
}

print(f"{n_actives} actives in {n_compounds} compounds "
      f"({n_actives / n_compounds:.2%}) - the kind of balance a screen has\n")

header = f"{'model':<26}{'accuracy':>10}{'precision':>11}{'recall':>8}{'F1':>7}{'MCC':>7}{'ROC-AUC':>9}{'PR-AUC':>8}{'hit@20':>8}"
print(header)
print("-" * len(header))

for name, s in scores.items():
    pred = (s >= 0.5).astype(int)                 # the yes/no decision
    top20 = np.argsort(-s, kind="stable")[:20]    # the ranking
    print(f"{name:<26}"
          f"{accuracy_score(y_mock, pred):>10.4f}"
          f"{precision_score(y_mock, pred, zero_division=0):>11.3f}"
          f"{recall_score(y_mock, pred, zero_division=0):>8.3f}"
          f"{f1_score(y_mock, pred, zero_division=0):>7.3f}"
          f"{matthews_corrcoef(y_mock, pred):>7.3f}"
          f"{roc_auc_score(y_mock, s):>9.3f}"
          f"{average_precision_score(y_mock, s):>8.3f}"
          f"{int(y_mock[top20].sum()):>8d}")

print()
print("Look down the accuracy column. Calling everything inactive scores 0.9982,")
print("which is HIGHER than the decent model - the only one of the three worth")
print("using. Every other column disagrees, and those are the ones telling the truth.")
print()
print("Compare ROC-AUC with PR-AUC as well. The random ranking sits near 0.5 on")
print("ROC, which sounds merely unimpressive, and near 0.00 on PR-AUC, which")
print("sounds like the disaster it is. Both numbers are correct.")

### The same point, drawn

Two views of the same predictions from the invented "decent model".

The **ROC** curve looks almost perfect. There are so many inactives that a few extra wrong
guesses barely change it, so it stays high even when the ranking is not very good.

The **precision-recall** curve is less forgiving. To find all 9 actives you have to accept a
lot of inactives along with them. The dashed line is what you would get by picking at
random, and it sits very low. That is why PR is the more useful view when actives are rare.

The **enrichment curve** answers the practical question: if I can only afford to test the
top few percent, how many of the actives do I get?

In [ ]:
# Still invented data - same helpers we will use on the real model later.
plot_pr_and_roc(y_mock, scores["A decent model"])
plt.show()

plot_enrichment_curve(y_mock, scores["A decent model"], zoom_frac=0.1)
plt.show()

print("Keep the shape of these in mind. The same two plots appear in Section 7,")
print("drawn from a model that was actually trained.")

---

# 🔁 Section 5 · Training the Model and Evaluating on Cross-Validation

## Cross-validation

We need to know how good the model is **before** we are allowed to open the test set. So we
work entirely inside the training data: cut it into five equal parts, called *folds*, and go
round five times.

```
round 1   [validate]   train      train      train      train
round 2    train      [validate]   train      train      train
round 3    train       train      [validate]   train      train
round 4    train       train       train      [validate]   train
round 5    train       train       train       train      [validate]
```

Each round fits a **fresh** model on four folds and scores it on the fifth. Every compound in
the training data is used for fitting four times, and for validation exactly once, so we get
five independent estimates instead of one lucky or unlucky split.

Note the vocabulary carefully: the held-out fold is a **validation fold**. It is not a test
set. The test set is a different file, and we have not opened it yet.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)


# Fit on the training folds, score on the held-out VALIDATION fold.
# Nothing in this cell touches the test data.
def train_and_validate(X_fold_train, X_fold_val, y_fold_train, y_fold_val):
    """Train a LightGBM model on the training folds and score it on the validation fold."""
    model = LGBMClassifier(random_state=42, verbose=-1)
    model.fit(X_fold_train, y_fold_train)

    y_pred = model.predict(X_fold_val)
    y_scores = model.predict_proba(X_fold_val)[:, 1]  # Probability for positive class

    metrics = {
        "Accuracy": accuracy_score(y_fold_val, y_pred),
        "Precision": precision_score(y_fold_val, y_pred, zero_division=0),
        "Recall": recall_score(y_fold_val, y_pred),
        "F1-Score": f1_score(y_fold_val, y_pred),
        "AUC-ROC": roc_auc_score(y_fold_val, y_scores) if len(set(y_fold_val)) > 1 else None,
        "MCC": matthews_corrcoef(y_fold_val, y_pred),
        "Cohen's Kappa": cohen_kappa_score(y_fold_val, y_pred),
    }

    # Hit@K: of the K highest-scoring compounds in this validation fold, how many are
    # active? Ks larger than the fold are skipped automatically.
    for k, hits in hits_at_k(y_fold_val, y_scores, ks=KS).items():
        metrics[f"Hits@{k}"] = hits

    return model, metrics


# Five-fold cross-validation, entirely within the training data
Nfold = 2
skf = StratifiedKFold(n_splits=Nfold, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(TrainData, TrainLabel)):
    # Four folds train the model, the fifth validates it
    X_fold_train, X_fold_val = TrainData[train_idx], TrainData[val_idx]
    y_fold_train = TrainLabel.iloc[train_idx]
    y_fold_val = TrainLabel.iloc[val_idx]

    _, metrics = train_and_validate(X_fold_train, X_fold_val, y_fold_train, y_fold_val)
    fold_metrics.append(metrics)

    print(f"Fold {fold_idx + 1} - validation metrics "
          f"({len(y_fold_train)} training rows, {len(y_fold_val)} validation rows):")
    for metric, value in metrics.items():
        # hit counts are integers; everything else is a rate
        print(f"{metric}: {value:d}" if metric.startswith("Hits@")
              else f"{metric}: {value:.4f}")
    print("-" * 100)

In [ ]:
# Average the validation metrics across the five folds
avg_metrics = {metric: np.mean([fold[metric] for fold in fold_metrics])
               for metric in fold_metrics[0]}

print("\nAverage validation metrics across all folds:")
for metric, value in avg_metrics.items():
    print(f"{metric}: {value:.1f}" if metric.startswith("Hits@")
          else f"{metric}: {value:.4f}")

# The validation folds are balanced 50/50, so hit@K comes out close to K - almost any
# compound you pick is active. That is exactly why these numbers say nothing about
# screening performance. The test set in Section 7 is where reality arrives.
plot_cv_metrics(fold_metrics)
plt.show()

---

# 🎛️ Section 6 · Hyperparameter Tuning

Some numbers a model *learns* from data. Others you have to *choose* before training starts,
how many trees, how fast each one corrects the last, how deep they grow. Those are
**hyperparameters**, and LightGBM's defaults are reasonable rather than right for your data.

## How people search

| Method | How it works | When to reach for it |
|---|---|---|
| **Grid search** | try every combination in a grid you define | few parameters, and you want exhaustive coverage |
| **Random search** | sample combinations at random from ranges you give | many parameters, usually finds something good faster than a grid, because most parameters do not matter much |
| **Bayesian optimisation** | build a model of which settings did well, then try where it predicts improvement | each fit is expensive and you can only afford a few dozen |

`scikit-learn` ships `GridSearchCV` and `RandomizedSearchCV`, both of which slot straight
around any estimator. Bayesian search needs a separate library, and **Optuna** is the usual
choice, with `scikit-optimize` and `hyperopt` as alternatives.

## What we do here

A small hand-picked grid of **5 configurations**, each scored by 5-fold cross-validation on
the training data. That is 50 model fits and takes about a minute.

Everything stays inside the training data. Tuning is a decision, and decisions are exactly
what the test set must not see, otherwise the estimate we get in Section 7 is no longer
trustworthy.

In [ ]:
# Five configurations, scored by 5-fold cross-validation on the TRAINING data only.
# 5 configs x 5 folds = 25 model fits. Each fit holds its own copy of the training fold, so
# the count is a memory budget as much as a time budget on a shared cluster.
from sklearn.metrics import roc_auc_score, average_precision_score

SELECT_K = 200  # we keep the configuration that puts the most actives in its top 200

param_grid = [
    {"n_estimators": 100, "learning_rate": 0.10, "num_leaves": 31},   # library defaults
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31},   # more trees
    {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31},   # slower learning
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 63},   # bushier trees
    {"n_estimators": 300, "learning_rate": 0.10, "num_leaves": 31, "colsample_bytree": 0.5},
]

tuning_cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

results = []
for config_id, params in enumerate(param_grid, start=1):
    hits, aucs, aps = [], [], []

    for train_idx, val_idx in tuning_cv.split(TrainData, TrainLabel):
        candidate = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1, **params)
        candidate.fit(TrainData[train_idx], TrainLabel.iloc[train_idx])

        y_val = TrainLabel.iloc[val_idx]
        scores = candidate.predict_proba(TrainData[val_idx])[:, 1]

        hits.append(hits_at_k(y_val, scores, ks=[SELECT_K])[SELECT_K])
        aucs.append(roc_auc_score(y_val, scores))
        aps.append(average_precision_score(y_val, scores))

    results.append({"config": config_id, **params,
                    f"Hits@{SELECT_K}": round(float(np.mean(hits)), 1),
                    "AUC-ROC": round(float(np.mean(aucs)), 4),
                    "avg precision": round(float(np.mean(aps)), 4)})
    print(f"  config {config_id:>2} of {len(param_grid)} done")

results_df = (pd.DataFrame(results)
              .fillna("-")
              .sort_values(f"Hits@{SELECT_K}", ascending=False))

print()
print(f"Ranked by mean Hits@{SELECT_K} across the five validation folds:")
print(results_df.to_string(index=False))

# The winner, and the settings we will carry into Section 7
best_config = int(results_df.iloc[0]["config"])
best_params = param_grid[best_config - 1]
best_score = results_df.iloc[0][f"Hits@{SELECT_K}"]
default_score = results_df.loc[results_df["config"] == 1, f"Hits@{SELECT_K}"].iloc[0]

print()
print(f"Best configuration : #{best_config}   Hits@{SELECT_K} = {best_score}")
print(f"  {best_params}")
print(f"Library defaults   : #1   Hits@{SELECT_K} = {default_score}")
print()
print("Section 7 trains the final model with these settings.")

---

# 📊 Section 7 · Train the Final Model

The choices are made. Section 5 showed the approach holds up under cross-validation and
Section 6 picked the hyperparameters, both using the training data alone.

Now retrain **one** model on the whole training set, with nothing held back, since there are
no decisions left for a validation fold to inform.

There is no test score to print here. The library has no labels, so the only verdict you get
is the leaderboard, and you get it after you submit.

In [ ]:
import joblib


def train_final_model(X, y, params):
    """Fit one model on ALL the training data, using the settings Section 6 chose."""
    final_model = LGBMClassifier(random_state=42, verbose=-1, n_jobs=-1, **params)
    final_model.fit(X, y)
    model_filename = RESULTS_DIR / "final_model.pkl"
    joblib.dump(final_model, model_filename)
    print(f"Final model saved as {model_filename}.")
    return final_model


print(f"Training the final model with the tuned settings: {best_params}")
final_model = train_final_model(TrainData, TrainLabel, best_params)

---

# 🎯 Section 8 · Rank the Library

Scoring the library *is* the virtual screen. Every compound gets a predicted probability of
being active, and the ranking those scores produce is your answer.

Nothing here can be checked. In the hands-on notebook the test set was labelled, so you could
see how many real actives landed near the top. Here you cannot, and neither could a real
screening campaign before it ran the assays.

In [ ]:
def evaluate_model(model, X):
    """Score a set of compounds and return the probability of being active."""
    return np.round(model.predict_proba(X)[:, 1], 3)


predictions = evaluate_model(final_model, TestData)

prediction_df = pd.DataFrame({
    'SMILES': df_test["SMILES"],
    'Prediction_Score': predictions,
})
prediction_df_sorted = prediction_df.sort_values(by='Prediction_Score', ascending=False)

nominees = prediction_df_sorted[prediction_df_sorted['Prediction_Score'] > 0.5]
print(f"Scored {len(prediction_df_sorted):,} compounds.")
print(f"Scoring above 0.5: {len(nominees):,}")
print(f"Your top 200 runs from {prediction_df_sorted['Prediction_Score'].iloc[0]:.3f} "
      f"down to {prediction_df_sorted['Prediction_Score'].iloc[199]:.3f}")
print()
print("Top 10:")
with pd.option_context("display.max_colwidth", 60):
    print(prediction_df_sorted.head(10).to_string(index=False))

---

# 📤 Section 9 · Submit Your Nominations

**This is the deliverable, and it is the last thing you do.** Sections 1 to 8 build the
model and rank the library; this writes the answer out.

Your submission is the **top 200 compounds** from your ranking, saved as a CSV with exactly
two columns:

| Column | What it holds |
|---|---|
| `SMILES` | the compound, as a SMILES string |
| `Prediction_Score` | your model's score for it |

**Rows must be ordered best first**, highest score at the top. The evaluation looks at how
near the top the real actives land, so the order is the answer, not just the set of 200. You
cannot check that yourself here, because the library carries no labels.

Two things to set in the next cell:

- `TEAM_NAME`, your team number: `Team1`, `Team2`, `Team3` and so on. The file is written
  as `<TEAM_NAME>.csv`.
- `SUBMISSION_DIR`, your own output folder. See the warning below.

> ⚠️ **Set `SUBMISSION_DIR` to your own folder before you run the cell.** Each person has
> their own volume on Azure and can only write to theirs, so the path ends in your user
> number: `output_user_05`, `output_user_09`, `output_user_23`. **Keep the leading zero**
> for numbers below ten: it is `output_user_05`, not `output_user_5`.
>
> The leaderboard reads every user folder every five minutes, so a file in the wrong
> place is not scored and a file in yours replaces whatever you sent before.


In [ ]:
# ---------------------------------------------------------------------------
# YOUR SUBMISSION - the top 200 compounds from your ranking
# ---------------------------------------------------------------------------
TEAM_NAME = "Team1"      # <-- CHANGE THIS to your team number: Team1, Team2, Team3 ...

# Write to your OWN volume. Everyone has one and can only write to theirs, so change 59
# to your user number, keeping the leading zero below ten: 05, 09, 23. A file written
# anywhere else is never scored.
SUBMISSION_DIR = Path("/Volumes/uhn_workshop/lab/output_user_59")

TOP_N = 200

ranked = prediction_df_sorted.loc[:, ["SMILES", "Prediction_Score"]]
submission = ranked.head(TOP_N).reset_index(drop=True)

# Check it before handing it in - a malformed file cannot be scored
assert list(submission.columns) == ["SMILES", "Prediction_Score"], "wrong columns"
assert len(submission) == TOP_N, f"expected {TOP_N} rows, got {len(submission)}"
assert submission["Prediction_Score"].is_monotonic_decreasing, "rows must be best-first"
assert submission["SMILES"].notna().all(), "every row needs a SMILES string"

submission_path = SUBMISSION_DIR / f"{TEAM_NAME}.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved {len(submission)} nominations")
print(f"  -> {submission_path}")
print()
print(submission.head())

---

## ⏱️ How long did that take?

Wall-clock time for the whole notebook, then every cell in the order it ran, and the slowest
five pulled out. Useful for planning a session, and for spotting a cell that is slower than
it looks.

To watch the timings live instead of waiting for this summary, set `ECHO_CELL_TIME = True` in
the first cell. Each cell then prints its own time underneath its output.

In [ ]:
elapsed = _time.time() - NOTEBOOK_STARTED
minutes, seconds = divmod(elapsed, 60)

print(f"Total run time : {int(minutes)} min {seconds:04.1f} s")
print(f"Cells executed : {len(CELL_TIMES)}")

if CELL_TIMES:
    measured = sum(t for t, _ in CELL_TIMES)
    print(f"Time in cells  : {measured:.1f} s "
          f"({measured / elapsed:.0%} of the total; the rest is start-up and idle time)")

    # every cell, in the order it ran
    print(f"\n{'cell':>4} {'seconds':>9} {'share':>7}  first line")
    print("-" * 78)
    for n, (taken, first_line) in enumerate(CELL_TIMES, start=1):
        share = taken / measured if measured else 0
        marker = " <-- slow" if taken >= 5 else ""
        print(f"{n:>4} {taken:>9.2f} {share:>6.1%}  {first_line}{marker}")

    print(f"\nSlowest five:")
    for taken, first_line in sorted(CELL_TIMES, reverse=True)[:5]:
        print(f"  {taken:6.1f}s  {first_line}")

print(f"\nMeasured on whatever machine ran this. Colab is usually slower than a laptop,")
print("so treat these as a guide rather than a promise.")
print("Set ECHO_CELL_TIME = True in the first cell to see each cell timed as it runs.")